<a href="https://colab.research.google.com/github/Leila828/age_gender_detector/blob/master/Names_gender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import requests
import time

# Load the names
df = pd.read_excel("names.xlsx")  # replace with your file
df["FirstName"] = df["Name"].apply(lambda x: str(x).strip().split()[0].capitalize())

# NamSor API setup
API_KEY = "your api here"
headers = {"X-API-KEY": API_KEY}
url = "https://v2.namsor.com/NamSorAPIv2/api2/json/gender"

# Function to query NamSor
def get_gender(first, last):
    try:
        response = requests.get(f"{url}/{first}/{last}", headers=headers)
        if response.status_code == 200:
            return response.json().get("likelyGender", "unknown")
        elif response.status_code == 429:
            print("Rate limit hit. Sleeping for 60 seconds...")
            time.sleep(60)
            return get_gender(first, last)
        else:
            print(f"Error {response.status_code} for {first}: {response.text}")
            return "unknown"
    except Exception as e:
        print(f"Exception for {first}: {e}")
        return "unknown"

# Avoid repeated calls for same first name
unique_firsts = df["FirstName"].unique()
gender_map = {}

for name in unique_firsts:
    gender = get_gender(name, name)  # Use first name as both first/last
    gender_map[name] = gender
    time.sleep(0.5)  # polite delay

# Map gender back
df["Gender"] = df["FirstName"].map(gender_map)
df.to_excel("names_with_gender.xlsx", index=False)
print("Done.")


Done.
